In [1]:
%pip install "pydantic[email]"
%pip install fastapi 

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [2]:
from enum import auto, IntFlag
from typing import Any

from pydantic import (
    BaseModel,
    EmailStr,
    Field,
    SecretStr,
    ValidationError,
)


# Define as permissões usando IntFlag, que permite combinações bitwise (ou seja, um usuário pode ter múltiplas roles)
class Role(IntFlag):
    Author = auto()
    Editor = auto()
    Developer = auto()
    Admin = Author | Editor | Developer

# Define o esquema de dados do Usuário herdando de BaseModel do Pydantic
class User(BaseModel):
    # Field ajuda a adicionar metadados, exemplos e restrições extras
    name: str = Field(examples=["Arjan"])
    # EmailStr valida automaticamente se a string segue o formato de um e-mail real
    # frozen=True impede que o valor do e-mail seja alterado após o objeto ser criado
    email: EmailStr = Field(
        examples=["example@arjancodes.com"],
        description="The email address of the user",
        frozen=True,
    )
    # SecretStr protege dados sensíveis: ao dar print no objeto, o valor aparece como '**********'
    password: SecretStr = Field(
        examples=["Password123"], description="The password of the user"
    )
    # Define a role padrão como None; o Pydantic converterá valores compatíveis para a Enum Role
    role: Role = Field(default=None, description="The role of the user")

# Função que tenta transformar um dicionário comum em um objeto da classe User
def validate(data: dict[str, Any]) -> None:
    try:
        # model_validate é o método que executa a validação rigorosa dos tipos e regras
        user = User.model_validate(data)
        print(user)
    except ValidationError as e:
        # Caso os dados não sigam o esquema (ex: e-mail inválido), o Pydantic gera erros detalhados
        print("User is invalid")
        for error in e.errors():
            print(error)

# Função principal para testar a validação com dados bons e ruins
def main() -> None:
    good_data = {
        "name": "Arjan",
        "email": "example@arjancodes.com",
        "password": "Password123",
    }
    bad_data = {"email": "<bad data>", "password": "<bad data>"}

    validate(good_data)
    validate(bad_data)


if __name__ == "__main__":
    main()

name='Arjan' email='example@arjancodes.com' password=SecretStr('**********') role=None
User is invalid
{'type': 'missing', 'loc': ('name',), 'msg': 'Field required', 'input': {'email': '<bad data>', 'password': '<bad data>'}, 'url': 'https://errors.pydantic.dev/2.12/v/missing'}
{'type': 'value_error', 'loc': ('email',), 'msg': 'value is not a valid email address: An email address must have an @-sign.', 'input': '<bad data>', 'ctx': {'reason': 'An email address must have an @-sign.'}}


In [3]:
import enum
import hashlib
import re
from typing import Any

from pydantic import (
    BaseModel,
    EmailStr,
    Field,
    field_validator,
    model_validator,
    SecretStr,
    ValidationError,
)

# Regex para senha: mínimo 8 caracteres, 1 maiúscula, 1 minúscula e 1 número
VALID_PASSWORD_REGEX = re.compile(r"^(?=.*[a-z])(?=.*[A-Z])(?=.*\d).{8,}$")
# Regex para nome: apenas letras e no mínimo 2 caracteres
VALID_NAME_REGEX = re.compile(r"^[a-zA-Z]{2,}$")


class Role(enum.IntFlag):
    Author = 1
    Editor = 2
    Admin = 4
    SuperAdmin = 8


class User(BaseModel):
    name: str = Field(examples=["Arjan"])
    email: EmailStr = Field(
        examples=["user@arjancodes.com"],
        description="The email address of the user",
        frozen=True,
    )
    password: SecretStr = Field(
        examples=["Password123"], description="The password of the user"
    )
    role: Role = Field(
        default=None, description="The role of the user", examples=[1, 2, 4, 8]
    )

    # Validador de campo específico: verifica o nome contra a Regex definida acima
    @field_validator("name")
    @classmethod
    def validate_name(cls, v: str) -> str:
        if not VALID_NAME_REGEX.match(v):
            raise ValueError(
                "Name is invalid, must contain only letters and be at least 2 characters long"
            )
        return v

    # Validador de campo com mode="before": processa o valor antes do Pydantic tentar converter para Role
    @field_validator("role", mode="before")
    @classmethod
    def validate_role(cls, v: int | str | Role) -> Role:
        # Dicionário de funções para converter Int, String ou manter como Role
        op = {int: lambda x: Role(x), str: lambda x: Role[x], Role: lambda x: x}
        try:
            return op[type(v)](v)
        except (KeyError, ValueError):
            # Se a string ou número não existir na Enum Role, levanta erro com as opções válidas
            raise ValueError(
                f"Role is invalid, please use one of the following: {', '.join([x.name for x in Role])}"
            )
        
    # Validador de modelo (múltiplos campos): analisa o dicionário de dados como um todo
    @model_validator(mode="before")
    @classmethod
    def validate_user(cls, v: dict[str, Any]) -> dict[str, Any]:
        # 1. Verifica se campos obrigatórios estão presentes
        if "name" not in v or "password" not in v:
            raise ValueError("Name and password are required")
        # 2. Regra de segurança: a senha não pode conter o nome do usuário (case-insensitive)
        if v["name"].casefold() in v["password"].casefold():
            raise ValueError("Password cannot contain name")
        # 3. Valida a força da senha usando a Regex
        if not VALID_PASSWORD_REGEX.match(v["password"]):
            raise ValueError(
                "Password is invalid, must contain 8 characters, 1 uppercase, 1 lowercase, 1 number"
            )
        # 4. Hashing: transforma a senha em um hash SHA-256 antes de criar o objeto
        v["password"] = hashlib.sha256(v["password"].encode()).hexdigest()
        return v


def validate(data: dict[str, Any]) -> None:
    try:
        user = User.model_validate(data)
        print(user)
    except ValidationError as e:
        print("User is invalid:")
        print(e)

# Conjunto de testes para cobrir todos os cenários de erro e sucesso
def main() -> None:
    test_data = dict(
        good_data={
            "name": "Arjan",
            "email": "example@arjancodes.com",
            "password": "Password123",
            "role": "Admin",
        },
        bad_role={
            "name": "Arjan",
            "email": "example@arjancodes.com",
            "password": "Password123",
            "role": "Programmer",
        },
        bad_data={
            "name": "Arjan",
            "email": "bad email",
            "password": "bad password",
        },
        bad_name={
            "name": "Arjan<-_->",
            "email": "example@arjancodes.com",
            "password": "Password123",
        },
        duplicate={
            "name": "Arjan",
            "email": "example@arjancodes.com",
            "password": "Arjan123",
        },
        missing_data={
            "email": "<bad data>",
            "password": "<bad data>",
        },
    )

    for example_name, data in test_data.items():
        print(example_name)
        validate(data)
        print()


if __name__ == "__main__":
    main()

good_data
name='Arjan' email='example@arjancodes.com' password=SecretStr('**********') role=<Role.Admin: 4>

bad_role
User is invalid:
1 validation error for User
role
  Value error, Role is invalid, please use one of the following: Author, Editor, Admin, SuperAdmin [type=value_error, input_value='Programmer', input_type=str]
    For further information visit https://errors.pydantic.dev/2.12/v/value_error

bad_data
User is invalid:
1 validation error for User
  Value error, Password is invalid, must contain 8 characters, 1 uppercase, 1 lowercase, 1 number [type=value_error, input_value={'name': 'Arjan', 'email'...ssword': 'bad password'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/value_error

bad_name
User is invalid:
1 validation error for User
name
  Value error, Name is invalid, must contain only letters and be at least 2 characters long [type=value_error, input_value='Arjan<-_->', input_type=str]
    For further information visit https://

In [4]:
import enum
import hashlib
import re
from typing import Any, Self
from pydantic import (
    BaseModel,
    EmailStr,
    Field,
    field_serializer,
    field_validator,
    model_serializer,
    model_validator,
    SecretStr,
)

# Configurações de Regex para validação de formato
VALID_PASSWORD_REGEX = re.compile(r"^(?=.*[a-z])(?=.*[A-Z])(?=.*\d).{8,}$")
VALID_NAME_REGEX = re.compile(r"^[a-zA-Z]{2,}$")


class Role(enum.IntFlag):
    User = 0
    Author = 1
    Editor = 2
    Admin = 4
    SuperAdmin = 8


class User(BaseModel):
    name: str = Field(examples=["Example"])
    email: EmailStr = Field(
        examples=["user@arjancodes.com"],
        description="The email address of the user",
        frozen=True,
    )
    # exclude=True garante que a senha nunca apareça no model_dump() ou exportação JSON
    password: SecretStr = Field(
        examples=["Password123"], description="The password of the user", exclude=True
    )
    # validate_default=True garante que o valor padrão (0) também passe pelos validadores
    role: Role = Field(
        description="The role of the user",
        examples=[1, 2, 4, 8],
        default=0,
        validate_default=True,
    )

    # Validador de campo: Checa se o nome segue o padrão de letras
    @field_validator("name")
    def validate_name(cls, v: str) -> str:
        if not VALID_NAME_REGEX.match(v):
            raise ValueError(
                "Name is invalid, must contain only letters and be at least 2 characters long"
            )
        return v

    # Validador de campo (Executa ANTES): Converte String/Int para a Enum Role
    @field_validator("role", mode="before")
    @classmethod
    def validate_role(cls, v: int | str | Role) -> Role:
        op = {int: lambda x: Role(x), str: lambda x: Role[x], Role: lambda x: x}
        try:
            return op[type(v)](v)
        except (KeyError, ValueError):
            raise ValueError(
                f"Role is invalid, please use one of the following: {', '.join([x.name for x in Role])}"
            )

    # Validador de modelo (Executa ANTES): Lógica de segurança e hashing da senha
    @model_validator(mode="before")
    @classmethod
    def validate_user_pre(cls, v: dict[str, Any]) -> dict[str, Any]:
        if "name" not in v or "password" not in v:
            raise ValueError("Name and password are required")
        if v["name"].casefold() in v["password"].casefold():
            raise ValueError("Password cannot contain name")
        if not VALID_PASSWORD_REGEX.match(v["password"]):
            raise ValueError(
                "Password is invalid, must contain 8 characters, 1 uppercase, 1 lowercase, 1 number"
            )
        # Transforma a senha em hash SHA-256
        v["password"] = hashlib.sha256(v["password"].encode()).hexdigest()
        return v

    # Validador de modelo (Executa DEPOIS): Valida regras de negócio com o objeto já criado
    @model_validator(mode="after")
    def validate_user_post(self, v: Any) -> Self:
        # Regra específica: Apenas um nome específico pode ser Admin
        if self.role == Role.Admin and self.name != "Arjan":
            raise ValueError("Only Arjan can be an admin")
        return self

    # Serializador de campo: Define que no JSON, a role deve aparecer como nome (ex: "Admin") e não número (4)
    @field_serializer("role", when_used="json")
    @classmethod
    def serialize_role(cls, v) -> str:
        return v.name

    # Serializador de modelo (mode="wrap"): Controla a saída final do dicionário/JSON
    @model_serializer(mode="wrap", when_used="json")
    def serialize_user(self, serializer, info) -> dict[str, Any]:
        # Se não houver filtros de inclusão/exclusão, retorna um formato simplificado
        if not info.include and not info.exclude:
            # Caso contrário, usa o comportamento padrão do Pydantic (respeitando o exclude da senha)
            return {"name": self.name, "role": self.role.name}
        return serializer(self)


def main() -> None:
    data = {
        "name": "Arjan",
        "email": "example@arjancodes.com",
        "password": "Password123",
        "role": "Admin",
    }

    # Cria o objeto validado
    user = User.model_validate(data)

    if user:
        # 1. model_dump(): Retorna um dicionário Python (objetos Python nativos)
        print(
            "The serializer that returns a dict:",
            user.model_dump(),
            sep="\n",
            end="\n\n",
        )
        # 2. model_dump(mode="json"): Retorna tipos compatíveis com JSON (usa o model_serializer customizado)
        print(
            "The serializer that returns a JSON string:",
            user.model_dump(mode="json"),
            sep="\n",
            end="\n\n",
        )
        # 3. Com exclusão: Ignora o model_serializer simplificado porque info.exclude está ativo
        print(
            "The serializer that returns a json string, excluding the role:",
            user.model_dump(exclude=["role"], mode="json"),
            sep="\n",
            end="\n\n",
        )
        # 4. dict(user): Conversão padrão do Python (exibe a senha como SecretStr)
        print("The serializer that encodes all values to a dict:", dict(user), sep="\n")


if __name__ == "__main__":
    main()

The serializer that returns a dict:
{'name': 'Arjan', 'email': 'example@arjancodes.com', 'role': <Role.Admin: 4>}

The serializer that returns a JSON string:
{'name': 'Arjan', 'role': 'Admin'}

The serializer that returns a json string, excluding the role:
{'name': 'Arjan', 'email': 'example@arjancodes.com'}

The serializer that encodes all values to a dict:
{'name': 'Arjan', 'email': 'example@arjancodes.com', 'password': SecretStr('**********'), 'role': <Role.Admin: 4>}


In [5]:
from datetime import datetime
from typing import Optional
from uuid import uuid4

from fastapi import FastAPI
from fastapi.responses import JSONResponse
from fastapi.testclient import TestClient
from pydantic import BaseModel, EmailStr, Field, field_serializer, UUID4

# Inicializa a aplicação FastAPI
app = FastAPI()


class User(BaseModel):
    # Configuração do modelo: "forbid" impede que o usuário envie campos extras não definidos
    model_config = {
        "extra": "forbid",
    }
    # Simulação de um banco de dados em memória (Lista estática)
    __users__ = []

    name: str = Field(..., description="Name of the user")
    email: EmailStr = Field(..., description="Email address of the user")
    
    # default_factory garante que cada nova instância receba uma lista nova e vazia
    friends: list[UUID4] = Field(
        default_factory=list, max_items=500, description="List of friends"
    )
    blocked: list[UUID4] = Field(
        default_factory=list, max_items=500, description="List of blocked users"
    )
    # kw_only=True obriga que esses campos sejam passados por nome, não por posição
    signup_ts: Optional[datetime] = Field(
        default_factory=datetime.now, description="Signup timestamp", kw_only=True
    )
    # Gera um UUID4 automaticamente para cada usuário criado
    id: UUID4 = Field(
        default_factory=uuid4, description="Unique identifier", kw_only=True
    )

    # Serializador para transformar o objeto UUID em String ao converter para JSON
    @field_serializer("id", when_used="json")
    def serialize_id(self, id: UUID4) -> str:
        return str(id)

# ROTA GET: Retorna a lista de usuários salvos
@app.get("/users", response_model=list[User])
async def get_users() -> list[User]:
    return list(User.__users__)

# ROTA POST: Cria um novo usuário. O FastAPI valida o JSON de entrada contra o modelo User
@app.post("/users", response_model=User)
async def create_user(user: User):
    User.__users__.append(user)
    return user

# ROTA GET: Busca um usuário específico pelo seu ID (UUID)
@app.get("/users/{user_id}", response_model=User)
async def get_user(user_id: UUID4) -> User | JSONResponse:
    try:
        # Busca o primeiro usuário com o ID correspondente
        return next((user for user in User.__users__ if user.id == user_id))
    except StopIteration:
        # Retorna erro 404 caso o ID não exista na lista
        return JSONResponse(status_code=404, content={"message": "User not found"})


def main() -> None:
    # TestClient simula chamadas HTTP para testar a API sem precisar subir o servidor
    with TestClient(app) as client:
        # 1. Testa a criação de 5 usuários
        for i in range(5):
            response = client.post(
                "/users",
                json={"name": f"User {i}", "email": f"example{i}@arjancodes.com"},
            )
            # Verifica se a criação foi bem-sucedida (Status 200 OK)
            assert response.status_code == 200
            assert response.json()["name"] == f"User {i}", (
                "The name of the user should be User {i}"
            )
            assert response.json()["id"], "The user should have an id"
            # Valida se os dados retornados batem com o esperado recriando o objeto via Pydantic
            user = User.model_validate(response.json())
            assert str(user.id) == response.json()["id"], "The id should be the same"
            assert user.signup_ts, "The signup timestamp should be set"
            assert user.friends == [], "The friends list should be empty"
            assert user.blocked == [], "The blocked list should be empty"

        # 2. Testa a listagem geral (deve ter 5 usuários agora)
        response = client.get("/users")
        assert response.status_code == 200, "Response code should be 200"
        assert len(response.json()) == 5, "There should be 5 users"
        
        # 3. Adiciona um 6º usuário para testar a busca individual logo em seguida
        response = client.post(
            "/users", json={"name": "User 5", "email": "example5@arjancodes.com"}
        )
        assert response.status_code == 200
        assert response.json()["name"] == "User 5", (
            "The name of the user should be User 5"
        )
        assert response.json()["id"], "The user should have an id"

        # Garante que o objeto retornado pelo servidor pode ser parseado corretamente pelo modelo
        user = User.model_validate(response.json())
        assert str(user.id) == response.json()["id"], "The id should be the same"
        assert user.signup_ts, "The signup timestamp should be set"
        assert user.friends == [], "The friends list should be empty"
        assert user.blocked == [], "The blocked list should be empty"

        # 4. Testa a busca de um usuário existente pelo seu ID dinâmico
        response = client.get(f"/users/{response.json()['id']}")
        assert response.status_code == 200
        assert response.json()["name"] == "User 5", (
            "This should be the newly created user"
        )

        # 5. Testa o cenário de erro: Busca por um UUID aleatório que não existe no "banco"
        response = client.get(f"/users/{uuid4()}")
        assert response.status_code == 404
        assert response.json()["message"] == "User not found", (
            "We technically should not find this user"
        )

        # 6. Testa a validação automática do FastAPI/Pydantic enviando um e-mail inválido
        # O código 422 (Unprocessable Entity) é retornado automaticamente quando a validação falha
        response = client.post("/users", json={"name": "User 6", "email": "wrong"})
        assert response.status_code == 422, "The email address is should be invalid"


if __name__ == "__main__":
    main()

C:\Users\danig\AppData\Local\Temp\ipykernel_3444\3693461857.py:26: PydanticDeprecatedSince20: `max_items` is deprecated and will be removed, use `max_length` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  friends: list[UUID4] = Field(
C:\Users\danig\AppData\Local\Temp\ipykernel_3444\3693461857.py:29: PydanticDeprecatedSince20: `max_items` is deprecated and will be removed, use `max_length` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  blocked: list[UUID4] = Field(
